# Minimal pulse diagrams

Matplotlib versions of the Echo, Lorentzian, and Square pulse diagrams.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


FIGURES_DIR = Path.cwd() / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

black = "#111111"
red = "#d71920"

# Symmetric pulse window: each pulse runs from -T to +T.
pulse_start_t = -1.0
pulse_end_t = 1.0
pulse_center_t = 0.0
measurement_t = 1.35

axis_start_t = -1.35
axis_end_t = 1.75
axis_start_x = 170
axis_end_x = 510


def time_to_x(t):
    return axis_start_x + (t - axis_start_t) / (axis_end_t - axis_start_t) * (axis_end_x - axis_start_x)


pulse_start_x = time_to_x(pulse_start_t)
pulse_end_x = time_to_x(pulse_end_t)
pulse_center_x = time_to_x(pulse_center_t)
measurement_x = time_to_x(measurement_t)
square_pulse_end_x = time_to_x(1.18)


def draw_time_axis(ax, y):
    ax.annotate(
        "",
        xy=(axis_end_x, y),
        xytext=(axis_start_x, y),
        arrowprops=dict(arrowstyle="-|>", lw=2.0, color=black, shrinkA=0, shrinkB=0),
    )
    ax.text(axis_end_x + 9, y, "t", fontsize=16, color=black, va="center", ha="left")


def add_measurement_square(ax, center_x, baseline_y, size=24):
    ax.add_patch(
        plt.Rectangle(
            (center_x - size / 2, baseline_y),
            size,
            size,
            facecolor=red,
            edgecolor="none",
        )
    )


def add_lorentzian_pulse(ax, baseline_y, height=50, tau_fraction=0.075):
    t = np.linspace(pulse_start_t, pulse_end_t, 300)
    x = np.array([time_to_x(value) for value in t])
    tau = (pulse_end_t - pulse_start_t) * tau_fraction
    envelope = height / (1 + ((t - pulse_center_t) / tau) ** 2)
    ax.fill_between(x, baseline_y, baseline_y + envelope, color=black, linewidth=0)


def add_signed_echo_lorentzian(ax, baseline_y, height=44, tau_fraction=0.075):
    t = np.linspace(pulse_start_t, pulse_end_t, 300)
    x = np.array([time_to_x(value) for value in t])
    tau = (pulse_end_t - pulse_start_t) * tau_fraction
    envelope = height / (1 + ((t - pulse_center_t) / tau) ** 2)
    signed_profile = -np.sign(t) * envelope
    signed_profile[np.isclose(t, 0)] = 0
    ax.fill_between(x, baseline_y, baseline_y + signed_profile, color=black, linewidth=0)


def add_square_pulse(ax, baseline_y, height=16):
    ax.add_patch(
        plt.Rectangle(
            (pulse_start_x, baseline_y),
            square_pulse_end_x - pulse_start_x,
            height,
            facecolor=black,
            edgecolor="none",
        )
    )


def setup_axis(figsize, ylim):
    fig, ax = plt.subplots(figsize=figsize, dpi=200)
    ax.set_xlim(0, 560)
    ax.set_ylim(*ylim)
    ax.axis("off")
    return fig, ax


def save_figure(fig, stem):
    png_path = FIGURES_DIR / f"{stem}.png"
    svg_path = FIGURES_DIR / f"{stem}.svg"
    fig.savefig(png_path, bbox_inches="tight", pad_inches=0.05, facecolor="white")
    fig.savefig(svg_path, bbox_inches="tight", pad_inches=0.05, facecolor="white")
    return png_path, svg_path


def draw_row(ax, label, y, pulse_fn):
    ax.text(36, y + 5, label, fontsize=14, color=black, va="center", ha="left")
    draw_time_axis(ax, y)
    pulse_fn(ax, y)
    add_measurement_square(ax, measurement_x, y)


combined_fig, combined_ax = setup_axis(figsize=(5.6, 3.2), ylim=(0, 330))
draw_row(combined_ax, "Echo", 255, add_signed_echo_lorentzian)
draw_row(combined_ax, "Lorentzian", 150, add_lorentzian_pulse)
draw_row(combined_ax, "Square", 55, add_square_pulse)

outputs = []
outputs.extend(save_figure(combined_fig, "minimal_pulse_diagram_combined"))

single_rows = [
    ("Echo", add_signed_echo_lorentzian, "minimal_pulse_diagram_echo"),
    ("Lorentzian", add_lorentzian_pulse, "minimal_pulse_diagram_lorentzian"),
    ("Square", add_square_pulse, "minimal_pulse_diagram_square"),
]

for label, pulse_fn, stem in single_rows:
    fig, ax = setup_axis(figsize=(5.6, 1.35), ylim=(0, 145))
    draw_row(ax, label, 70, pulse_fn)
    outputs.extend(save_figure(fig, stem))

plt.show()

outputs